# 06 — Interobserver Agreement (IOA)

Computes IOA between a primary observer and a secondary observer for the same session.

**Methods supported**:
- **Exact agreement** (discrete events): proportion of events both observers agree occurred
- **Time-window agreement** (10-second bins): both observers log at least one event in same bin
- **Total count IOA**: smaller count / larger count × 100 (for rate DVs)

**Convention**: flag sessions with IOA < 80% for review.

## How to prepare IOA files

1. Run the companion app with the **primary observer** for a session → exports
   `P001_LVL001_ABCD1234_20240101_120000.csv`
2. Run the companion app independently with the **secondary observer** for the
   same session (simultaneous observation) → exports a second CSV with a
   different observer_id.
3. Place both files in `data/raw/`. The pipeline will separate them by `observer_id`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

ROOT     = Path('..').resolve()
RAW_DIR  = ROOT / 'data' / 'raw'
FIG_DIR  = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

IOA_THRESHOLD = 80.0  # flag sessions below this
WINDOW_S      = 10.0  # seconds per bin for time-window IOA

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})


## 1. Load all raw CSVs and find sessions with multiple observers


In [ ]:
META_CODES = {'SESSION_PAUSE', 'SESSION_RESUME', 'SESSION_END'}

frames = []
for f in sorted(RAW_DIR.glob('*.csv')):
    df = pd.read_csv(f, dtype=str)
    if 'observer_id' in df.columns:
        df['elapsed_s'] = pd.to_numeric(df['elapsed_s'], errors='coerce')
        frames.append(df)

if not frames:
    print('No CSV files found in data/raw/. Run sessions first.')
else:
    all_data = pd.concat(frames, ignore_index=True)
    beh = all_data[~all_data['event_code'].isin(META_CODES)].copy()

    # Find sessions with >1 unique observer
    ioa_sessions = (
        beh.groupby(['participant_id', 'phenomenon', 'level_id'])['observer_id']
           .nunique()
           .reset_index()
           .query('observer_id > 1')
    )

    if ioa_sessions.empty:
        print('No sessions with multiple observers found.')
        print('To compute IOA: collect data for the same session with two observers')
        print('using different observer_id values in the companion app setup.')
    else:
        print(f'Sessions with multiple observers: {len(ioa_sessions)}')
        display(ioa_sessions)


## 2. Compute IOA — three methods


In [ ]:
def total_count_ioa(n1: int, n2: int) -> float:
    """Smaller / larger × 100."""
    if max(n1, n2) == 0:
        return 100.0
    return min(n1, n2) / max(n1, n2) * 100


def time_window_ioa(df1: pd.DataFrame, df2: pd.DataFrame,
                    event_code: str, window_s: float = 10.0) -> float:
    """Proportion of windows where both observers agree (both coded ≥1 event
    of event_code, or both coded 0)."""
    max_t = max(df1['elapsed_s'].max(), df2['elapsed_s'].max())
    bins  = np.arange(0, max_t + window_s, window_s)
    if len(bins) < 2:
        return np.nan

    e1 = df1[df1['event_code'] == event_code]['elapsed_s'].values
    e2 = df2[df2['event_code'] == event_code]['elapsed_s'].values

    c1, _ = np.histogram(e1, bins=bins)
    c2, _ = np.histogram(e2, bins=bins)

    agree = np.sum((c1 > 0) == (c2 > 0))
    return agree / len(c1) * 100


def exact_event_ioa(codes1: list, codes2: list) -> float:
    """Proportion of events in the shorter list that match the longer list.
    Uses sequence alignment (edit-distance approach)."""
    matches = sum(a == b for a, b in zip(codes1, codes2))
    total   = max(len(codes1), len(codes2))
    return matches / total * 100 if total > 0 else 100.0


if 'ioa_sessions' in dir() and not ioa_sessions.empty:
    ioa_results = []

    for _, sess_row in ioa_sessions.iterrows():
        pid     = sess_row['participant_id']
        phenom  = sess_row['phenomenon']
        level   = sess_row['level_id']

        subset = beh[(beh['participant_id'] == pid) &
                     (beh['phenomenon']     == phenom) &
                     (beh['level_id']       == level)]
        observers = subset['observer_id'].unique()

        for obs1, obs2 in combinations(observers, 2):
            df1 = subset[subset['observer_id'] == obs1].sort_values('elapsed_s')
            df2 = subset[subset['observer_id'] == obs2].sort_values('elapsed_s')

            # Primary DV event for this phenomenon
            PRIMARY_EVENTS = {
                'positive_reinforcement': 'RESPONSE',
                'reinforcement_schedules': 'RESPONSE',
                'extinction': 'RESPONSE',
                'negative_reinforcement': 'ESCAPE',
                'shaping': 'SR_APPROX',
                'stimulus_control': 'RESP_SD_PLUS',
                'generalization': 'RESPONSE',
                'punishment': 'TARGET_RESP',
                'chaining': 'CHAIN_COMPLETE',
                'matching_law': 'LEFT',
                'explore_exploit': 'EXPLORE',
            }
            primary_ev = PRIMARY_EVENTS.get(phenom, 'RESPONSE')

            n1 = (df1['event_code'] == primary_ev).sum()
            n2 = (df2['event_code'] == primary_ev).sum()

            tc_ioa  = total_count_ioa(n1, n2)
            tw_ioa  = time_window_ioa(df1, df2, primary_ev, WINDOW_S)
            seq_ioa = exact_event_ioa(
                df1[df1['event_code'] == primary_ev]['event_code'].tolist(),
                df2[df2['event_code'] == primary_ev]['event_code'].tolist()
            )

            ioa_results.append({
                'participant_id': pid,
                'phenomenon':     phenom,
                'level_id':       level,
                'observer_pair':  f'{obs1} / {obs2}',
                'primary_event':  primary_ev,
                'n_obs1':         n1,
                'n_obs2':         n2,
                'total_count_IOA': round(tc_ioa, 1),
                'time_window_IOA': round(tw_ioa, 1) if not np.isnan(tw_ioa) else np.nan,
                'below_threshold': tc_ioa < IOA_THRESHOLD,
            })

    ioa_df = pd.DataFrame(ioa_results)
    print(f'=== IOA Results (threshold = {IOA_THRESHOLD}%) ===')
    display(ioa_df)

    flagged = ioa_df[ioa_df['below_threshold']]
    if flagged.empty:
        print(f'\nAll sessions meet the {IOA_THRESHOLD}% threshold ✓')
    else:
        print(f'\n⚠ {len(flagged)} session(s) below {IOA_THRESHOLD}% — review required:')
        display(flagged[['participant_id', 'phenomenon', 'observer_pair',
                          'total_count_IOA', 'time_window_IOA']])


## 3. IOA summary chart


In [ ]:
if 'ioa_df' in dir() and not ioa_df.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(ioa_df))
    colors = ['#e63946' if b else '#2a9d8f' for b in ioa_df['below_threshold']]
    ax.bar(x, ioa_df['total_count_IOA'], color=colors, alpha=0.85)
    ax.axhline(IOA_THRESHOLD, ls='--', color='black', lw=1.5,
                label=f'{IOA_THRESHOLD}% threshold')
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{r['participant_id']}\n{r['phenomenon'][:8]}" for _, r in ioa_df.iterrows()],
        rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Total Count IOA (%)')
    ax.set_ylim(0, 110)
    ax.set_title('IOA by Session (red = below threshold)')
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'ioa_summary.png', bbox_inches='tight')
    plt.show()

    # Save IOA table
    ioa_df.to_csv(ROOT / 'data' / 'processed' / 'ioa_results.csv', index=False)
    print('IOA results saved to data/processed/ioa_results.csv')
